# 06 - Market Basket Analysis

Objetivo: analizar afinidades entre productos comprados juntos para proponer recomendaciones, packs comerciales y promociones cruzadas.

Este notebook usa las tablas `dwh.product_affinity` y `dwh.product_recommendations`, generadas por `sql/06_market_basket.sql`.

In [ ]:
from pathlib import Path
import os

import pandas as pd
from sqlalchemy import create_engine, text

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DB_NAME = os.getenv("DB_NAME", "saleshealth")
DB_USER = os.getenv("DB_USER", os.getenv("USER", "postgres"))
DB_HOST = os.getenv("DB_HOST", "")
DB_PORT = os.getenv("DB_PORT", "")
DB_PASSWORD = os.getenv("DB_PASSWORD", "")

if DB_HOST:
    auth = f"{DB_USER}:{DB_PASSWORD}@" if DB_PASSWORD else f"{DB_USER}@"
    port = f":{DB_PORT}" if DB_PORT else ""
    DATABASE_URL = os.getenv("DATABASE_URL", f"postgresql+psycopg2://{auth}{DB_HOST}{port}/{DB_NAME}")
else:
    DATABASE_URL = os.getenv("DATABASE_URL", f"postgresql+psycopg2:///{DB_NAME}")

engine = create_engine(DATABASE_URL)

def q(sql):
    return pd.read_sql_query(text(sql), engine)

In [ ]:
affinity = q("select * from dwh.product_affinity")
recommendations = q("select * from dwh.product_recommendations")
affinity.shape, recommendations.shape

In [ ]:
affinity[[
    "baskets_together",
    "support",
    "confidence_a_to_b",
    "confidence_b_to_a",
    "lift",
    "avg_basket_revenue",
]].describe().round(4)

In [ ]:
q('''
select
    product_name,
    recommended_product_name,
    baskets_together,
    confidence,
    lift,
    support,
    recommendation_rank
from dwh.product_recommendations
order by lift desc, confidence desc
limit 20;
''')

In [ ]:
q('''
select
    product_name,
    count(*) as recomendaciones,
    round(avg(lift)::numeric, 4) as lift_medio,
    round(max(lift)::numeric, 4) as lift_maximo
from dwh.product_recommendations
group by product_name
order by lift_maximo desc, recomendaciones desc
limit 15;
''')

In [ ]:
top = recommendations.sort_values(["lift", "confidence"], ascending=[False, False]).head(15)
ax = top.plot(
    x="recommended_product_name",
    y="lift",
    kind="bar",
    figsize=(10, 4),
    title="Top recomendaciones por lift",
    legend=False,
)
ax.set_xlabel("Producto recomendado")
ax.set_ylabel("Lift")

In [ ]:
q('''
select
    product_a_name,
    product_b_name,
    baskets_together,
    confidence_a_to_b,
    confidence_b_to_a,
    lift,
    avg_basket_revenue
from dwh.product_affinity
where lift > 1
order by lift desc, baskets_together desc
limit 20;
''')

Lectura de negocio:

- `support` indica en que porcentaje de tickets aparece el par.
- `confidence` indica la probabilidad de recomendar B cuando se compra A.
- `lift` mide si la combinacion aparece mas de lo esperable por azar.
- Reglas con `lift > 1` son candidatas para packs o venta cruzada.